# Imports and Environment Setup

In [ ]:
# Import all required packages
import os
import shutil
import json
import glob
import random
from pathlib import Path
from collections import defaultdict

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import Image, display
from ultralytics import YOLO

# Check environment
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

HOME = os.getcwd()
print(f"Working directory: {HOME}")

# Install Dependencies

# Install required packages
!pip install ultralytics roboflow supervision flash-attn opencv-python pillow

# If you need the specific YOLOv12 implementation
!pip install git+https://github.com/sunsmarterjie/yolov12.git

# Configuration

Set `DOMAIN`, then run the cells top to bottom. Run **Train** once per domain; **Inference** only needs the trained weights.

In [ ]:
# Pick the domain; every path below follows from it.
DOMAIN = "aircraft"  # "aircraft" or "car"

# Jupyter starts in pipeline/detection/, but the rest of the pipeline runs from the
# project root. Walk up to the folder that holds configs/ and data/.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "configs").is_dir() and (p / "data").is_dir())

# Roboflow "YOLOv12" export (train/, valid/, test/, data.yaml). Not in the repo:
# unzip the export for each domain into datasets/<domain>/.
DATA_YAML_PATH = PROJECT_ROOT / "datasets" / DOMAIN / "data.yaml"

# Training runs and the copied weights (gitignored).
RUNS_DIR = PROJECT_ROOT / "runs" / "detect"
OUTPUT_MODEL_DIR = PROJECT_ROOT / "trained_models" / f"yolov12_{DOMAIN}"
MODEL_PATH = OUTPUT_MODEL_DIR / "weights" / "best.pt"

# Inference runs on the pipeline's test images and writes the JSON that
# configs/<domain>.yaml (paths.detections_json) and pipeline/ensemble.py read.
SOURCE_FOLDER = PROJECT_ROOT / "data" / DOMAIN / "images"
JSON_OUTPUT_PATH = PROJECT_ROOT / "data" / DOMAIN / "detections" / f"yolov12_{DOMAIN}.json"
ANNOTATED_FOLDER = PROJECT_ROOT / "results" / DOMAIN / "detections_annotated"

# Settings recorded in the shipped data/<domain>/detections/ files
# (car: YOLOv12m, conf 0.25, IoU 0.2; aircraft: conf 0.5, IoU left at the default).
MODEL_CFG = {"aircraft": "yolov12l.yaml", "car": "yolov12m.yaml"}[DOMAIN]
CONFIDENCE_THRESHOLD = {"aircraft": 0.5, "car": 0.25}[DOMAIN]
IOU_THRESHOLD = {"aircraft": 0.7, "car": 0.2}[DOMAIN]

print(f"Project root: {PROJECT_ROOT}")
print(f"Domain: {DOMAIN} | model: {MODEL_CFG} | conf: {CONFIDENCE_THRESHOLD} | IoU: {IOU_THRESHOLD}")

# Path verification

In [ ]:
# Verify your paths exist
paths_to_check = [
    DATA_YAML_PATH,    # training only
    MODEL_PATH,        # inference only; created by the Train cell
    SOURCE_FOLDER,
]

for path in paths_to_check:
    exists = path.exists()
    print(f"{'✅' if exists else '❌'} {path}")
    if exists and path.is_dir():
        files = list(path.glob('*'))
        print(f"   Contains {len(files)} items")

# Train

In [ ]:
os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)

print(f"Starting YOLOv12 training ({DOMAIN}, {MODEL_CFG})...")

# Load model
model = YOLO(MODEL_CFG)

# Train model
results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=200,
    imgsz=640,
    patience=15,
    project=str(RUNS_DIR),
    name=DOMAIN
)

print("Training completed!")

# Ultralytics appends a number when the run folder already exists (aircraft2, ...),
# so copy the folder this run actually wrote.
train_dir = Path(model.trainer.save_dir)
shutil.copytree(train_dir, OUTPUT_MODEL_DIR, dirs_exist_ok=True)
print(f"Model copied from {train_dir} to: {OUTPUT_MODEL_DIR}")

# Inference 

In [ ]:
os.makedirs(ANNOTATED_FOLDER, exist_ok=True)
os.makedirs(JSON_OUTPUT_PATH.parent, exist_ok=True)

# Load model
print("Loading model...")
model = YOLO(str(MODEL_PATH))
class_names = model.names
print(f"Model loaded. Class names: {class_names}")

# Supported image extensions
img_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
image_paths = sorted(p for p in SOURCE_FOLDER.iterdir() if p.suffix.lower() in img_extensions)

print(f"Found {len(image_paths)} images. Running inference with confidence threshold: {CONFIDENCE_THRESHOLD}")

# === Run inference ===
results_list = []

for img_path in image_paths:
    print(f"Processing: {img_path.name}")
    results = model.predict(source=str(img_path), save=False, conf=CONFIDENCE_THRESHOLD, iou=IOU_THRESHOLD)
    result = results[0]
    
    components = defaultdict(int)
    confidences = []
    detections = []
    
    if result.boxes is not None and len(result.boxes) > 0:
        boxes = result.boxes
        for i in range(len(boxes)):
            conf = float(boxes.conf[i].cpu().numpy())
            class_id = int(boxes.cls[i].cpu().numpy())
            class_name = class_names[class_id]
            
            bbox = boxes.xyxy[i].cpu().numpy().tolist()
            
            components[class_name] += 1
            confidences.append(conf)
            
            detections.append({
                "class": class_name,
                "confidence": round(conf, 4),
                "bbox": [round(coord, 2) for coord in bbox]
            })
    
    avg_conf = float(np.mean(confidences)) if confidences else 0.0
    if components:
        print(f"  Detected components: {dict(components)} | Avg confidence: {avg_conf:.2f}")
    else:
        print("  No objects detected above threshold.")
    
    # Visualize detections
    annotated = result.plot()
    cv2.imwrite(str(ANNOTATED_FOLDER / img_path.name), annotated)
    
    # pipeline/ensemble.py reads "average_confidence" (see docs/DETECTION_SETUP.md).
    image_result = {
        "image_id": img_path.name,
        "components": dict(components),
        "average_confidence": round(avg_conf, 2),
        "total_detections": len(detections),
        "detections": detections
    }
    results_list.append(image_result)

# Save JSON file
final_output = {
    "model": f"YOLOv12 ({MODEL_CFG})",
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "iou_threshold": IOU_THRESHOLD,
    "total_images": len(image_paths),
    "total_detections": sum(r["total_detections"] for r in results_list),
    "results": results_list
}

with open(JSON_OUTPUT_PATH, "w") as f:
    json.dump(final_output, f, indent=2)

print(f"\n✅ Inference complete.")
print(f"Annotated images saved to: {ANNOTATED_FOLDER}")
print(f"JSON summary saved to: {JSON_OUTPUT_PATH}")